In [7]:
#!/usr/bin/env python
"""
Interactive UMAP and t-SNE visualization with group/label co-projection.
Loads embeddings by label (exact species) or group (genus-level).
If selection is a group (e.g., 'aedes'): displays 'Aedes spp.' in title,
assigns undefined points species names from group members (e.g., 'Ae. aegypti').
If selection is a label (e.g., 'gambiae'): displays exact species in title (e.g., 'An. gambiae'),
all points assigned that single species. Supports manual/HDBSCAN clustering, selection via lasso,
rename/delete clusters, and export as Nature-ready PDF/PNG with separate legends (horizontal 4-col, no titles).
"""
###################################### imports
from datetime import datetime
import time
import resource
import os
import hdbscan
import numpy as np
import pandas as pd
from pyprojroot import here
from ipywidgets import Dropdown, Button, HBox, VBox, IntSlider, Output, Label, Checkbox, Text, ColorPicker
from IPython.display import display, clear_output
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
###################################### constants
BASE_DIR = "data/dim_red"
CONFIG_PATH = "config/query_configs.tsv"
MARKER_SIZE = 3
DEFAULT_COLOR = "#8C8C8C"
ALPHA = 0.55
PALETTE_NATURE = ["#e76f51", "#f4a261", "#e9c46a", "#2a9d8f", "#4F6F7C", "#264653"]
MIN_CLUSTER_SIZE = 800
FIGURE_WIDTH = 900
FIGURE_HEIGHT = 500
SQUARE_SIZE = 8.0
ZOOM_FACTOR_UMAP = 0.9
ZOOM_FACTOR_TSNE = 1.10
TICK_FONT_SIZE = 13
TICK_FONT_FAMILY = "DejaVu Sans"
TICK_DECIMALS = 1
N_TICKS = 6
LEGEND_FONT_SIZE = 13
TITLE_FONT_SIZE = 15
DPI = 300
LEGEND_NCOL = 4
###################################### state
start_time = time.time()
config_df = pd.read_csv(str(here() / CONFIG_PATH), sep="\t")
if "genus" in config_df.columns and "label" in config_df.columns:
    config_df["exact_species"] = config_df.apply(lambda row: f"{str(row['genus'])[0]}. {row['label']}" if not str(row['exact_species']).startswith(str(row['genus'])[0]) else row['exact_species'], axis=1)
halys_row = pd.DataFrame([{"proteome_id": "UP_HALYS", "order": "Hemiptera", "genus": "Halyomorpha", "exact_species": "H. halys", "label": "halys", "ncbi_taxonomy": "85300", "group": "stinkbug"}])
if "halys" not in config_df["label"].values: config_df = pd.concat([config_df, halys_row], ignore_index=True)
groups_dict = config_df.groupby("group")["label"].apply(list).to_dict()
all_labels = sorted(config_df["label"].unique().tolist())
all_groups = sorted(config_df["group"].unique().tolist())
select_options = [""] + sorted(set(all_labels + all_groups))
label_to_species = config_df.set_index("label")["exact_species"].to_dict()
select_dropdown = Dropdown(options=select_options, description="Select:")
checklist_box = HBox([])
mcs_slider = IntSlider(value=MIN_CLUSTER_SIZE, min=100, max=10000, step=10, description="MinCS:", continuous_update=True, layout={"width": "320px"})
fit_on_dropdown = Dropdown(options=["umap", "tsne"], value="umap", description="Fit on:", layout={"width": "150px"})
cluster_name = Text(value="NewCluster", description="Name:", layout={"width": "180px"})
cluster_color = ColorPicker(value="#E7298A", description="Color:", layout={"width": "160px"})
assign_btn = Button(description="Assign selected", button_style="primary")
rename_dropdown = Dropdown(options=[], description="Rename:", layout={"width": "160px"})
rename_text = Text(value="", description="To:", layout={"width": "140px"})
rename_btn = Button(description="Rename", button_style="info")
reset_btn = Button(description="Reset", button_style="danger")
delete_btn = Button(description="Delete", button_style="danger")
plot_fp_text = Text(value="", description="Export dir:", layout={"width": "420px"})
output_area = Output()
plot_container = VBox([])
state = {"ids_umap": None, "umap_coords": None, "tsne_coords": None, "cluster_labels": None, "cluster_info": {}, "filtered_mask": None, "current_select": None, "label_source": None, "checkboxes": {}, "fig_widget": None, "selected_indices": [], "last_selection_source": "umap", "display_title": ""}
###################################### functions
def load_npz(basename, method="umap"):
    base_dir = str(here() / BASE_DIR)
    file_path = f"{base_dir}/{basename}_{method}.npz"
    with np.load(file_path) as data:
        ids = data["ids"]
        mat_key = next((k for k in data.files if k != "ids"), None)
        if mat_key is None: raise ValueError(f"No embedding found in {file_path}")
        embedding = data[mat_key]
    return ids, embedding
def fit_hdbscan_auto(coords, min_cluster_size):
    coords = coords.astype(np.float64)
    mcs = int(min_cluster_size)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=min(10, mcs), metric="euclidean", gen_min_span_tree=True, prediction_data=True)
    raw_labels = clusterer.fit_predict(coords)
    valid_mask = raw_labels != -1
    valid_labels = raw_labels[valid_mask]
    if len(valid_labels) > 0:
        unique_labs, counts = np.unique(valid_labels, return_counts=True)
        sorted_labs = unique_labs[np.argsort(-counts)]
        MAX_CLUSTERS = min(15, len(sorted_labs))
        rank_map = {old: rank+1 for rank, old in enumerate(sorted_labs[:MAX_CLUSTERS])}
        labels = np.array([rank_map.get(lab, 0) for lab in raw_labels])
        n_clusters = len(rank_map)
    else:
        labels, n_clusters = np.zeros(len(coords), dtype=int), 0
    return labels, n_clusters
def update_checklist(labels_list):
    checklist_box.children = []
    state["checkboxes"] = {}
    if not labels_list: return
    boxes = []
    for lbl in labels_list:
        display_lbl = label_to_species.get(lbl, lbl)
        cb = Checkbox(value=True, description=display_lbl, indent=False)
        cb.observe(lambda c: update_visibility(), names="value")
        state["checkboxes"][lbl] = cb
        boxes.append(cb)
    checklist_box.children = boxes
def update_visibility():
    if state["fig_widget"] is None or state["label_source"] is None: return
    active = {label_to_species.get(lbl, lbl) for lbl, cb in state["checkboxes"].items() if cb.value}
    mask = np.array([lbl in active for lbl in state["label_source"]])
    state["filtered_mask"] = mask
    alphas = np.where(mask, ALPHA, 0.02)
    with state["fig_widget"].batch_update():
        for trace in list(state["fig_widget"].data)[:2]:
            trace.marker.opacity = alphas
def refresh_cluster_dropdown():
    opts = [f"{cid}: {info['name']}" for cid, info in sorted(state["cluster_info"].items())]
    rename_dropdown.options = opts
    if opts: rename_dropdown.value = opts[0]
def update_plot_traces():
    if state["umap_coords"] is None or state["cluster_labels"] is None or state["fig_widget"] is None: return
    fw = state["fig_widget"]
    colors = [state["cluster_info"].get(lab, {"color": DEFAULT_COLOR})["color"] for lab in state["cluster_labels"]]
    names = [state["cluster_info"].get(lab, {"name": "unassigned"})["name"] for lab in state["cluster_labels"]]
    texts = [f"{src} ({nam})" for src, nam in zip(state["label_source"], names)]
    with fw.batch_update():
        for i in range(2):
            fw.data[i].marker.color = colors
            fw.data[i].text = texts
        while len(fw.data) > 2: fw.data = list(fw.data)[:2]
        unique_species = list(dict.fromkeys(state["label_source"].tolist()))
        for sp in unique_species:
            n = int(np.sum(state["label_source"] == sp))
            fw.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color=DEFAULT_COLOR), name=f"{sp} (n={n})", legendgroup=sp, showlegend=True))
        cluster_ids = sorted([cid for cid in state["cluster_info"] if cid != 0], key=lambda c: -np.sum(state["cluster_labels"] == c))
        for cid in cluster_ids:
            info = state["cluster_info"][cid]
            n = int(np.sum(state["cluster_labels"] == cid))
            fw.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color=info["color"]), name=f"{info['name']} (n={n})", legendgroup=info["name"], showlegend=True))
def on_selection_umap(trace, points, selector):
    if points.point_inds:
        state["selected_indices"] = list(points.point_inds)
        state["last_selection_source"] = "umap"
        with output_area:
            clear_output(wait=True)
            print(f"Selected {len(state['selected_indices'])} points (UMAP view).")
def on_selection_tsne(trace, points, selector):
    if points.point_inds:
        state["selected_indices"] = list(points.point_inds)
        state["last_selection_source"] = "tsne"
        with output_area:
            clear_output(wait=True)
            print(f"Selected {len(state['selected_indices'])} points (t-SNE view).")
def assign_selected(_):
    with output_area:
        clear_output(wait=True)
        if not state["selected_indices"] or state["cluster_labels"] is None:
            print("Select points first (and fit or load)")
            return
        existing_ids = [cid for cid in state["cluster_info"] if cid != 0]
        new_id = max(existing_ids) + 1 if existing_ids else 1
        prefix = "U" if state["last_selection_source"] == "umap" else "T"
        default_name = f"{prefix}-{new_id:02d}"
        name = cluster_name.value.strip()
        if not name or name == "NewCluster": name = default_name
        color = cluster_color.value
        state["cluster_labels"][state["selected_indices"]] = new_id
        state["cluster_info"][new_id] = {"name": name, "color": color}
        update_plot_traces()
        refresh_cluster_dropdown()
        print(f"Assigned {len(state['selected_indices'])} pts to cluster {new_id} '{name}'")
        state["selected_indices"] = []
def rename_cluster(_):
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value:
            print("No cluster selected")
            return
        cid = int(rename_dropdown.value.split(":")[0])
        new_name = rename_text.value.strip()
        if not new_name:
            print("Enter a new name")
            return
        if cid in state["cluster_info"]:
            state["cluster_info"][cid]["name"] = new_name
            update_plot_traces()
            refresh_cluster_dropdown()
            print(f"Renamed cluster {cid} to '{new_name}'")
def reset_clusters(_):
    with output_area:
        clear_output(wait=True)
        if state["umap_coords"] is None:
            print("Load data first")
            return
        state["cluster_labels"] = np.zeros(len(state["ids_umap"]), dtype=int)
        state["cluster_info"] = {0: {"name": "unassigned", "color": DEFAULT_COLOR}}
        update_plot_traces()
        refresh_cluster_dropdown()
        print("Reset all cluster assignments to unassigned.")
def delete_cluster(_):
    with output_area:
        clear_output(wait=True)
        if not rename_dropdown.value:
            print("No cluster selected")
            return
        cid = int(rename_dropdown.value.split(":")[0])
        if cid == 0:
            print("Cannot delete unassigned cluster")
            return
        if cid in state["cluster_info"]:
            state["cluster_labels"][state["cluster_labels"] == cid] = 0
            del state["cluster_info"][cid]
            update_plot_traces()
            refresh_cluster_dropdown()
            print(f"Deleted cluster {cid} and unassigned its points.")
def load_and_plot(_):
    with output_area:
        clear_output(wait=True)
        sel = select_dropdown.value
        if not sel:
            print("Select a label or group")
            return
        print(f"Loading '{sel}'...")
        try:
            is_group = sel in groups_dict
            u_ids, u_coords = load_npz(sel, "umap")
            _, t_coords = load_npz(sel, "tsne")
            if len(u_ids) != len(u_coords) or len(u_ids) != len(t_coords):
                print("Dimension mismatch between ids and coordinates")
                return
            if is_group:
                group_labels = groups_dict[sel]
                l_source_all = ["unknown"] * len(u_ids)
                id_to_species = {}
                for lbl in group_labels:
                    try:
                        lbl_ids, _ = load_npz(lbl, "umap")
                        sp_name = label_to_species.get(lbl, lbl)
                        for uid in lbl_ids: id_to_species[uid] = sp_name
                    except: pass
                for idx, uid in enumerate(u_ids):
                    if uid in id_to_species: l_source_all[idx] = id_to_species[uid]
                label_list_for_box = group_labels
                display_title = sel.capitalize() + " spp."
            else:
                sp_name = label_to_species.get(sel, sel)
                l_source_all = [sp_name] * len(u_ids)
                label_list_for_box = []
                display_title = sp_name
            state["display_title"] = display_title
            state["ids_umap"] = np.array(u_ids)
            state["umap_coords"] = u_coords
            state["tsne_coords"] = t_coords
            state["label_source"] = np.array(l_source_all)
            state["current_select"] = sel
            state["filtered_mask"] = np.ones(len(u_ids), dtype=bool)
            state["selected_indices"] = []
            state["cluster_labels"] = np.zeros(len(u_ids), dtype=int)
            state["cluster_info"] = {0: {"name": "unassigned", "color": DEFAULT_COLOR}}
            update_checklist(label_list_for_box)
            refresh_cluster_dropdown()
            date_str = datetime.now().strftime("%Y%m%d_%H%M%S")
            plot_fp_text.value = str(here() / "viz" / f"{sel}_{date_str}")
            initial_colors = [DEFAULT_COLOR] * len(u_ids)
            fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{display_title} - UMAP", f"{display_title} - t-SNE"), horizontal_spacing=0.08, column_widths=[0.5, 0.5])
            trace1 = go.Scattergl(x=state["umap_coords"][:, 0], y=state["umap_coords"][:, 1], mode="markers", marker=dict(size=MARKER_SIZE, color=initial_colors, opacity=ALPHA), text=state["label_source"], customdata=state["ids_umap"], name="UMAP Data", showlegend=False)
            trace2 = go.Scattergl(x=state["tsne_coords"][:, 0], y=state["tsne_coords"][:, 1], mode="markers", marker=dict(size=MARKER_SIZE, color=initial_colors, opacity=ALPHA), text=state["label_source"], customdata=state["ids_umap"], name="t-SNE Data", showlegend=False)
            fig.add_trace(trace1, row=1, col=1)
            fig.add_trace(trace2, row=1, col=2)
            unique_species = list(dict.fromkeys(state["label_source"].tolist()))
            for sp in unique_species:
                n = int(np.sum(state["label_source"] == sp))
                fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color=DEFAULT_COLOR), name=f"{sp} (n={n})", legendgroup=sp, showlegend=True))
            fig.update_layout(height=FIGURE_HEIGHT, width=FIGURE_WIDTH, showlegend=True, dragmode="lasso", hovermode="closest", plot_bgcolor="white", paper_bgcolor="white", margin=dict(l=55, r=40, t=55, b=85), legend=dict(orientation="h", y=-0.16, x=0.5, xanchor="center", yanchor="top", bgcolor="rgba(0,0,0,0)"), font=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="#222222"), title_font=dict(size=TITLE_FONT_SIZE, family=TICK_FONT_FAMILY, color="#111111"))
            fig.update_annotations(font=dict(size=TITLE_FONT_SIZE, family=TICK_FONT_FAMILY, color="#111111"))
            for col_idx, coords, zf in [(1, state["umap_coords"], ZOOM_FACTOR_UMAP), (2, state["tsne_coords"], ZOOM_FACTOR_TSNE)]:
                xmin, xmax = float(np.min(coords[:, 0])), float(np.max(coords[:, 0]))
                ymin, ymax = float(np.min(coords[:, 1])), float(np.max(coords[:, 1]))
                mx, my = (xmin + xmax) / 2, (ymin + ymax) / 2
                hx = (xmax - xmin) * zf / 2
                hy = (ymax - ymin) * zf / 2
                x_range = [mx - hx, mx + hx]
                y_range = [my - hy, my + hy]
                tickvals_x = np.linspace(x_range[0], x_range[1], N_TICKS)
                ticktext_x = [f"{v:.{TICK_DECIMALS}f}" for v in tickvals_x]
                tickvals_y = np.linspace(y_range[0], y_range[1], N_TICKS)
                ticktext_y = [f"{v:.{TICK_DECIMALS}f}" for v in tickvals_y]
                fig.update_xaxes(range=x_range, tickvals=tickvals_x, ticktext=ticktext_x, tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="#333333"), ticks="outside", ticklen=5, tickwidth=1.2, tickcolor="#444444", linecolor="#444444", linewidth=1.2, side="bottom", mirror=False, showgrid=False, zeroline=False, row=1, col=col_idx)
                fig.update_yaxes(range=y_range, tickvals=tickvals_y, ticktext=ticktext_y, tickfont=dict(size=TICK_FONT_SIZE, family=TICK_FONT_FAMILY, color="#333333"), ticks="outside", ticklen=5, tickwidth=1.2, tickcolor="#444444", linecolor="#444444", linewidth=1.2, side="left", mirror=False, showgrid=False, zeroline=False, row=1, col=col_idx)
            fw = go.FigureWidget(fig)
            fw.data[0].on_selection(on_selection_umap)
            fw.data[1].on_selection(on_selection_tsne)
            state["fig_widget"] = fw
            plot_container.children = [fw]
            print(f"Loaded {len(u_ids)} points for '{display_title}'. Use lasso tool to select points.")
        except Exception as e:
            print(f"Error: {e}")
def fit_clusters(_):
    with output_area:
        clear_output(wait=True)
        if state["umap_coords"] is None:
            print("Load data first")
            return
        mcs = int(mcs_slider.value)
        fit_on = fit_on_dropdown.value
        coords = state["umap_coords"] if fit_on == "umap" else state["tsne_coords"]
        print(f"Fitting HDBSCAN on {fit_on.upper()} with MinCS={mcs}...")
        labels, n_cl = fit_hdbscan_auto(coords, mcs)
        state["cluster_labels"] = labels.copy()
        state["cluster_info"] = {0: {"name": "unassigned", "color": DEFAULT_COLOR}}
        prefix = "U" if fit_on == "umap" else "T"
        for i in range(1, n_cl + 1):
            state["cluster_info"][i] = {"name": f"{prefix}-{i:02d}", "color": PALETTE_NATURE[(i - 1) % len(PALETTE_NATURE)]}
        update_plot_traces()
        refresh_cluster_dropdown()
        print(f"{fit_on.upper()} clusters fitted: {n_cl} clusters found")
def export_results(_):
    with output_area:
        clear_output(wait=True)
        if state["ids_umap"] is None or state["cluster_labels"] is None:
            print("Load and fit/assign first")
            return
        selected_mask = np.zeros(len(state["ids_umap"]), dtype=bool)
        if state["selected_indices"]: selected_mask[state["selected_indices"]] = True
        else: selected_mask = state["filtered_mask"] if state["filtered_mask"] is not None else np.ones(len(state["ids_umap"]), dtype=bool)
        names = [state["cluster_info"].get(lab, {"name": "unassigned"})["name"] for lab in state["cluster_labels"]]
        df_export = pd.DataFrame({"id": state["ids_umap"], "cluster_id": state["cluster_labels"], "cluster_name": names, "select": state["current_select"], "label": state["label_source"], "in_selection": selected_mask})
        date_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        out_path = here() / "viz" / f"{state['current_select']}_clusters_{date_str}.tsv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        df_export.to_csv(out_path, sep="\t", index=False)
        print(f"Exported to {out_path}")
def export_plot(_):
    with output_area:
        clear_output(wait=True)
        if state["fig_widget"] is None or state["umap_coords"] is None:
            print("Load data first")
            return
        fp = plot_fp_text.value.strip()
        if not fp:
            print("Specify export directory")
            return
        try:
            out_dir = here() / fp if not (fp.startswith("/") or (len(fp) > 1 and fp[1] == ":")) else fp
            out_dir = str(out_dir)
            os.makedirs(out_dir, exist_ok=True)
            colors = np.array([state["cluster_info"].get(lab, {"color": DEFAULT_COLOR})["color"] for lab in state["cluster_labels"]])
            fig, axes = plt.subplots(1, 2, figsize=(SQUARE_SIZE, SQUARE_SIZE * 0.5), dpi=DPI)
            for ax, coords, title_suffix, zf in zip(axes, [state["umap_coords"], state["tsne_coords"]], ["UMAP", "t-SNE"], [ZOOM_FACTOR_UMAP, ZOOM_FACTOR_TSNE]):
                ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=MARKER_SIZE, alpha=ALPHA, linewidths=0, rasterized=True)
                xmin, xmax = float(np.min(coords[:, 0])), float(np.max(coords[:, 0]))
                ymin, ymax = float(np.min(coords[:, 1])), float(np.max(coords[:, 1]))
                mx, my = (xmin + xmax) / 2, (ymin + ymax) / 2
                hx = (xmax - xmin) * zf / 2
                hy = (ymax - ymin) * zf / 2
                ax.set_xlim(mx - hx, mx + hx)
                ax.set_ylim(my - hy, my + hy)
                ax.set_title(f"{state['display_title']} - {title_suffix}", fontsize=TITLE_FONT_SIZE, fontname=TICK_FONT_FAMILY, color="#111111")
                ax.tick_params(axis="both", which="major", labelsize=TICK_FONT_SIZE, length=5, width=1.2, colors="#333333")
                for spine in ax.spines.values():
                    spine.set_color("#444444")
                    spine.set_linewidth(1.2)
                ax.set_facecolor("white")
                ax.grid(False)
            fig.patch.set_facecolor("white")
            plt.tight_layout()
            fig.savefig(f"{out_dir}/plot_square_nolegend.pdf", dpi=DPI, bbox_inches="tight", facecolor="white")
            fig.savefig(f"{out_dir}/plot_square_nolegend.png", dpi=DPI, bbox_inches="tight", facecolor="white")
            plt.close(fig)
            legend_items = []
            unique_species = list(dict.fromkeys(state["label_source"].tolist()))
            for sp in unique_species:
                n = int(np.sum(state["label_source"] == sp))
                legend_items.append((f"{sp} (n={n})", DEFAULT_COLOR))
            cluster_ids = sorted([cid for cid in state["cluster_info"] if cid != 0], key=lambda c: -np.sum(state["cluster_labels"] == c))
            for cid in cluster_ids:
                info = state["cluster_info"][cid]
                n = int(np.sum(state["cluster_labels"] == cid))
                legend_items.append((f"{info['name']} (n={n})", info["color"]))
            handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=col, markersize=10, label=name) for name, col in legend_items]
            fig_h = plt.figure(figsize=(14, 1.5), dpi=DPI)
            ax_h = fig_h.add_subplot(111)
            ax_h.axis("off")
            leg_h = ax_h.legend(handles=handles, loc="center", ncol=LEGEND_NCOL, frameon=False, prop={"family": TICK_FONT_FAMILY, "size": LEGEND_FONT_SIZE})
            fig_h.canvas.draw()
            renderer = fig_h.canvas.get_renderer()
            bbox_h = leg_h.get_window_extent(renderer).transformed(fig_h.dpi_scale_trans.inverted())
            fig_h.patch.set_facecolor("white")
            fig_h.savefig(f"{out_dir}/legend_horizontal.pdf", dpi=DPI, bbox_inches=bbox_h, pad_inches=0.02, facecolor="white")
            fig_h.savefig(f"{out_dir}/legend_horizontal.png", dpi=DPI, bbox_inches=bbox_h, pad_inches=0.02, facecolor="white")
            plt.close(fig_h)
            fig_v = plt.figure(figsize=(3, 8), dpi=DPI)
            ax_v = fig_v.add_subplot(111)
            ax_v.axis("off")
            leg_v = ax_v.legend(handles=handles, loc="center", ncol=1, frameon=False, prop={"family": TICK_FONT_FAMILY, "size": LEGEND_FONT_SIZE})
            fig_v.canvas.draw()
            renderer_v = fig_v.canvas.get_renderer()
            bbox_v = leg_v.get_window_extent(renderer_v).transformed(fig_v.dpi_scale_trans.inverted())
            fig_v.patch.set_facecolor("white")
            fig_v.savefig(f"{out_dir}/legend_vertical.pdf", dpi=DPI, bbox_inches=bbox_v, pad_inches=0.02, facecolor="white")
            fig_v.savefig(f"{out_dir}/legend_vertical.png", dpi=DPI, bbox_inches=bbox_v, pad_inches=0.02, facecolor="white")
            plt.close(fig_v)
            print(f"Exported Nature-ready files to {out_dir}/ : plot_square_nolegend.{{pdf,png}} + legend_horizontal.{{pdf,png}} + legend_vertical.{{pdf,png}} (DPI={DPI})")
        except Exception as e:
            print(f"Export failed: {e}")
###################################### ui
load_btn = Button(description="Load", button_style="info")
fit_btn = Button(description="Fit HDBSCAN", button_style="success")
export_btn = Button(description="Export TSV", button_style="warning")
export_plot_btn = Button(description="Export PDF/PNG", button_style="warning")
load_btn.on_click(load_and_plot)
fit_btn.on_click(fit_clusters)
export_btn.on_click(export_results)
export_plot_btn.on_click(export_plot)
assign_btn.on_click(assign_selected)
rename_btn.on_click(rename_cluster)
reset_btn.on_click(reset_clusters)
delete_btn.on_click(delete_cluster)
top_row = HBox([Label("Select label/group:"), select_dropdown, load_btn])
checklist_row = HBox([Label("Toggle labels:"), checklist_box])
cluster_row = HBox([Label("Clustering:"), mcs_slider, fit_on_dropdown, fit_btn, export_btn])
manual_row = HBox([Label("Manual cluster:"), cluster_name, cluster_color, assign_btn])
rename_row = HBox([Label("Rename:"), rename_dropdown, rename_text, rename_btn, reset_btn, delete_btn])
export_plot_row = HBox([Label("Plot export:"), plot_fp_text, export_plot_btn])
controls = VBox([top_row, checklist_row, cluster_row, manual_row, rename_row, export_plot_row, output_area, plot_container])
display(controls)
###################################### execution
executiontime = time.time() - start_time
try: maxmemory = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
except: maxmemory = 0
date_logged = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"[PYTHON-INFO] {executiontime:.2f} {maxmemory} {date_logged}")

[PYTHON-INFO] 0.04 456236 2026-09-18 10:00:04
